# 1. Análise de Anomalias Estruturais dos Microdados ENEM (2009–2025)

**Objetivo:** Este notebook tem como objetivo analisar a estrutura dos arquivos de microdados disponibilizados pelo INEP no período de 2009 a 2025, verificando se houve mudanças no padrão de organização e distribuição dos dados ao longo desses anos.

A análise se concentra exclusivamente nas características estruturais dos arquivos, sem considerar, neste momento, a seleção de features ou a definição de variáveis-alvo (targets). Os resultados servirão de base para definir a estratégia de ingestão e processamento dos dados na arquitetura do pipeline de Big Data, que será desenvolvida nas próximas etapas do projeto.

In [1]:
import pandas as pd
import os
import glob

RAW_DATA_DIR = '../data/raw'

# Mapeamento estrutural: para cada ano, identificar quantos arquivos CSV existem
# e se o arquivo monolítico principal (MICRODADOS_ENEM_XXXX) está presente.

anos_dirs = sorted(glob.glob(os.path.join(RAW_DATA_DIR, 'enem_*')))

resultado = []

for dir_path in anos_dirs:
    ano = os.path.basename(dir_path).split('_')[1]
    csv_files = glob.glob(os.path.join(dir_path, '**', '*.csv'), recursive=True)
    # Filtrar arquivos relevantes (ignorar Dicionarios e Zone.Identifier)
    csv_relevantes = [f for f in csv_files if not any(x in f for x in ['Zone.Identifier', 'Dicionario', 'dicionario'])]
    nomes = [os.path.basename(f) for f in csv_relevantes]

    tem_monolitico = any(n.upper().startswith('MICRODADOS_ENEM') for n in nomes)
    tem_participantes = any('PARTICIPANTES' in n.upper() for n in nomes)
    tem_resultados = any('RESULTADOS' in n.upper() for n in nomes)

    if tem_monolitico:
        formato = "✅ Monolítico (1 arquivo)"
        status = "✅ Dentro do padrão histórico"
    elif tem_participantes and tem_resultados:
        formato = "⚠️ Bipartido (PARTICIPANTES + RESULTADOS)"
        status = "❌ RUPTURA ESTRUTURAL"
    elif tem_participantes:
        formato = "⚠️ Parcial (só PARTICIPANTES)"
        status = "❌ RUPTURA ESTRUTURAL"
    else:
        formato = "❓ Desconhecido"
        status = "❓ Verificar manualmente"

    resultado.append({
        'Ano': ano,
        'Qtd Arquivos CSV': len(csv_relevantes),
        'Arquivos Encontrados': ', '.join(nomes),
        'Formato': formato,
        'Status': status
    })

df = pd.DataFrame(resultado)
display(df[['Ano', 'Qtd Arquivos CSV', 'Formato', 'Status']])

,Ano,Qtd Arquivos CSV,Formato,Status
0,2009,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
1,2010,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
2,2011,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
3,2012,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
4,2013,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
5,2014,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
6,2015,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
7,2016,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
8,2017,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico
9,2018,2,✅ Monolítico (1 arquivo),✅ Dentro do padrão histórico


## 1. O Padrão Monolítico Histórico (2009–2023)

Historicamente, o INEP disponibilizou os microdados do ENEM em um único arquivo tabular para cada ano de aplicação, seguindo uma estrutura semelhante à apresentada abaixo:

```text
data/raw/enem_XXXX/microdados_enem_XXXX/DADOS/MICRODADOS_ENEM_XXXX.csv
```

Esses arquivos, que podem ocupar vários gigabytes, reúnem em uma única tabela as informações dos participantes, incluindo dados cadastrais, notas das provas e respostas ao questionário socioeconômico.

Conforme apresentado na tabela anterior, esse formato foi mantido nas edições de 2009 a 2023. A padronização da estrutura ao longo desse período facilita a integração das bases históricas e permite estabelecer uma estratégia de processamento conjunto dos dados no pipeline de Big Data.


## 2. A Ruptura Estrutural de 2024 e 2025

A partir do ENEM 2024, o INEP abandonou o formato monolítico. Os dados passaram a ser entregues em **dois arquivos separados**:

| Arquivo | Conteúdo |
|---|---|
| `PARTICIPANTES_XXXX.csv` | Dados de identificação, localização da prova e respostas do questionário socioeconômico (Q001–Q023) |
| `RESULTADOS_XXXX.csv` | Notas por área de conhecimento e redação |
| `ITENS_PROVA_XXXX.csv` | Parâmetros psicometricos dos itens de prova — **desprezível para o nosso modelo** |


In [8]:
# Verificar a estrutura dos arquivos bipartidos de 2024 e fail-fast
path_part = os.path.join(RAW_DATA_DIR, 'enem_2024/microdados_enem_2024/DADOS/PARTICIPANTES_2024.csv')
path_res  = os.path.join(RAW_DATA_DIR, 'enem_2024/microdados_enem_2024/DADOS/RESULTADOS_2024.csv')

try:
    df_part = pd.read_csv(path_part, sep=';', encoding='latin-1', nrows=5)
    print(f"PARTICIPANTES_2024.csv → {len(df_part.columns)} colunas")
    print()
    df_res = pd.read_csv(path_res,  sep=';', encoding='latin-1', nrows=5)
    print(f"RESULTADOS_2024.csv   → {len(df_res.columns)} colunas")
    print()

    # Prova do mecanismo fail-fast
    import subprocess
    n_part = int(subprocess.check_output(['wc', '-l', path_part]).split()[0]) - 1
    n_res  = int(subprocess.check_output(['wc', '-l', path_res]).split()[0])  - 1
    print(f"Checagem:\nPARTICIPANTES = {n_part:,} linhas \nRESULTADOS    = {n_res:,} linhas")
    print(f"Linhas iguais? {n_part == n_res} → {'JOIN posicional seguro' if n_part == n_res else 'ALERTA: divergência detectada!'}")

except FileNotFoundError as e:
    print(f"Arquivo não encontrado: {e}")

PARTICIPANTES_2024.csv → 38 colunas

RESULTADOS_2024.csv   → 42 colunas

Checagem:
PARTICIPANTES = 4,332,944 linhas 
RESULTADOS    = 4,332,944 linhas
Linhas iguais? True → JOIN posicional seguro


## 3. Conclusão: A Estratégia para 2024/2025

A análise estrutural comprova que:
1. De **2009 a 2023**, o INEP manteve um padrão monolítico consistente e confiável.
2. Em **2024 e 2025**, houve uma ruptura para um formato bipartido (`PARTICIPANTES` e `RESULTADOS`), onde o vínculo por chave primária não está disponível no segundo arquivo.

**Decisão Arquitetural:**
Para lidar com a ruptura de 2024/2025 sem corromper o dataset, será criado um módulo
para efetuar uma concatenacao entre os dois arquivos com uma checagem de seguranca que verifica previamente se os arquivos possuem a mesma quantidade de linhas (provado empiricamente na célula acima). Caso haja divergência, o pipeline é interrompido.